In [1]:
## importing libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import kruskal
from statsmodels.stats.multitest import multipletests
from scipy.stats import mannwhitneyu
from cliffs_delta import cliffs_delta




In [2]:
df = pd.read_excel('finalforms.xlsx')

In [3]:
# pd.set_option('display.max_columns', None)
pd.set_option('display.max_columns',None)

In [4]:
print(df.shape)
# # print(df.head())
# df.columns[18]

(62, 104)


In [5]:
# Clean column names: strip spaces, replace line breaks, compress spaces
df.columns = (
    df.columns
    .str.replace("\n", " ", regex=False)
    .str.replace("\r", " ", regex=False)
    .str.strip()
    .str.replace(" +", " ", regex=True)
)

In [6]:
df["Voice-based system (robot / assistant)"].value_counts()

Voice-based system (robot / assistant)
I have not used    53
I have used         5
Name: count, dtype: int64

In [7]:
df["Digital screen (tablet / kiosk / QR code / app)"].value_counts()

Digital screen (tablet / kiosk / QR code / app)
I have used     58
Name: count, dtype: int64

In [8]:
df["Human waiter"].value_counts()

Human waiter
I have used     58
Name: count, dtype: int64

In [9]:
digital_likert_cols = [
"I rarely need help from staff when using digital systems",
"It is easy to know how to begin ordering using a digital system",
"It is easy to share my seating preferences when booking digitally",
"It is easy to customize my order when using a digital system",
"If something goes wrong, I can easily correct or seek help through the system",
"Wait time for table",
"Wait time for food (meals)",
"Wait time for bill",
"I feel confident about what to do next when ordering through a digital menu",
"I feel confident finding dishes that fit my diet or allergies",
"I feel confident making payments through a digital system",
"I feel comfortable paying without staff involvement",
"I prefer to ask staff only for special or complex requests rather than routine actions"
]


In [10]:
# 

LIKERT_ORDER = {
    "strongly disagree": 1,
    "disagree": 2,
    "slightly disagree": 3,
    "slightly agree": 4,
    "agree": 5,
    "strongly agree": 6,
}


In [11]:
LIKERT_LABELS = {
    1: "strongly disagree",
    2: "disagree",
    3: "slightly disagree",
    4: "slightly agree",
    5: "agree",
    6: "strongly agree",
}


In [12]:
unique_responses = set()

for col in digital_likert_cols:
    unique_responses.update(
        df[col]
        .dropna()
        .astype(str)
        .str.strip()
        .str.lower()
        .unique()
    )

sorted(unique_responses)




['agree',
 'disagree',
 'slightly agree',
 'slightly disagree',
 'strongly agree',
 'strongly disagree']

In [13]:
scale_keys = set(LIKERT_ORDER.keys())

unexpected = unique_responses - scale_keys
missing = scale_keys - unique_responses

unexpected, missing


(set(), set())

In [14]:
for col in digital_likert_cols:
    df[col] = (
        df[col]
        .astype(str)
        .str.strip()
        .str.lower()
    )


In [15]:
for col in digital_likert_cols:
    df[col] = df[col].map(LIKERT_ORDER)


In [16]:
# df_human_answered = df[df[likert_cols].notna().any(axis=1)]

df_digital = df[df[digital_likert_cols].notna().any(axis=1)].copy()



In [17]:
mask_any = df[digital_likert_cols].notna().any(axis=1)

n_digital = mask_any.sum()

print("Number of respondents who answered the digital section:", n_digital)

Number of respondents who answered the digital section: 25


In [18]:
# df_digital.shape

In [19]:
df_digital["Age band"].value_counts()

Age band
25–34    19
35–44     4
18–24     2
Name: count, dtype: int64

In [20]:
# # groups

df_digital["Age band"].unique()



array(['25–34', '35–44', '18–24'], dtype=object)

In [21]:
# df["Age band"].unique()

In [22]:
df_digital.shape

(25, 104)

In [23]:
age_counts = (
    df_digital["Age band"]
    .value_counts(dropna=False)
    .sort_index()
)

age_counts

Age band
18–24     2
25–34    19
35–44     4
Name: count, dtype: int64

In [24]:
def merge_age_band(age):
    if pd.isna(age):
        return np.nan
    elif age in ["18–24", "25–34"]:
        return "18–34"
    elif age in ["35–44", "45–54"]:
        return "35–54"
    elif age in ["55–64", "65+"]:
        return "55+"
    else:
        return np.nan

df_digital = df_digital.copy()

df_digital["Age_3grp"] = df_digital["Age band"].apply(merge_age_band)


In [25]:
df_digital["Age_3grp"]

1     18–34
5     18–34
9     35–54
11    18–34
13    35–54
15    18–34
16    18–34
19    18–34
21    18–34
22    18–34
25    18–34
29    18–34
30    18–34
32    18–34
37    18–34
38    18–34
39    35–54
41    18–34
47    18–34
49    18–34
50    18–34
57    35–54
58    18–34
59    18–34
61    18–34
Name: Age_3grp, dtype: object

In [26]:
df_digital["Age_3grp"].value_counts()

Age_3grp
18–34    21
35–54     4
Name: count, dtype: int64

In [27]:

# -------------------------------------------------------------------
# Digital interaction age-group comparison
#
# This analysis uses the filtered dataset `df_digital`, which contains
# only respondents who answered at least one digital interaction item.
#
# The merged age variable `Age_3grp` is used to define two groups:
#   - 18–34
#   - 35–54
#
# For each question in `digital_likert_cols`:
#   1. Responses are extracted separately for the two age groups.
#   2. A Mann–Whitney U test is performed to assess whether the
#      distribution of Likert-scale responses differs between groups.
#   3. Cliff’s delta is calculated as an effect size to quantify the
#      magnitude of the difference between the two groups.
#
# The resulting statistics are stored in lists:
#   - U_vals: Mann–Whitney U statistics
#   - p_vals: p-values from the Mann–Whitney tests
#   - delta_vals: Cliff’s delta effect sizes
#   - delta_size: qualitative interpretation of the effect size
#
# These lists are later combined into a results dataframe for reporting.
# -------------------------------------------------------------------

p_vals = []
U_vals = []
delta_vals = []
delta_size = []

age_levels = sorted(df_digital["Age_3grp"].dropna().unique())

for q in digital_likert_cols:

    g1 = df_digital.loc[df_digital["Age_3grp"] == age_levels[0], q].dropna()
    g2 = df_digital.loc[df_digital["Age_3grp"] == age_levels[1], q].dropna()

    if len(g1) > 0 and len(g2) > 0:
        U, p = mannwhitneyu(g1, g2, alternative="two-sided")
        delta, size = cliffs_delta(g1, g2)
    else:
        U, p, delta, size = np.nan, np.nan, np.nan, None

    U_vals.append(U)
    p_vals.append(p)
    delta_vals.append(delta)
    delta_size.append(size)

In [ ]:
p_vals